<a href="https://colab.research.google.com/github/qjqj7890/Predict-Customer-Churn/blob/main/Predict_Customer_Churn(Kaggle).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install catboost
!pip install optuna

In [16]:
!mkdir -p /etc/OpenCL/vendors && echo "libnvidia-opencl.so.1" > /etc/OpenCL/vendors/nvidia.icd
!pip install catboost==1.2.2

  Using cached catboost-1.2.2.tar.gz (60.1 MB)
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [17]:
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [18]:
train_raw = pd.read_csv('train.csv')
test_raw = pd.read_csv('test.csv')

In [19]:
orig_url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
orig_raw = pd.read_csv(orig_url)
orig_raw

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


# EDA

In [20]:
train_raw = train_raw.dropna().reset_index(drop=True)

In [21]:
# Robust Data Cleaning & Feature Engineering 함수 정의
def clean_and_engineer(df):
    df = df.copy()

    # TotalCharges 숫자 변환 및 공백 처리
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

    # Feature Engineering (Service Intensity)
    services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
    for s in services:
        df[s] = df[s].fillna('No')

    # 이용 중인 서비스 개수 합산
    df['ServiceCount'] = (df[services] == 'Yes').sum(axis=1)

    # 월 요금 / 총 요금 비율
    df['MonthlyToTotalRatio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)

    # 병합 및 인코딩을 위해 범주형 변수들을 str 타입으로 통일
    cat_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
                'MultipleLines', 'InternetService', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
                'PaperlessBilling', 'PaymentMethod']
    for col in cat_cols:
        df[col] = df[col].astype(str)

    return df

In [22]:
# 전처리 함수 적용
df = clean_and_engineer(train_raw)
test = clean_and_engineer(test_raw)
orig = clean_and_engineer(orig_raw)

In [23]:
# Target Map (문자열 'Yes'/'No'를 1/0으로 변환)
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})
orig['Churn'] = orig['Churn'].map({'No': 0, 'Yes': 1})

In [24]:
# 2. Merge Original Dataset Probabilities (버그 수정 버전)
encode_cols = ['Contract', 'InternetService', 'OnlineSecurity', 'tenure', 'MonthlyCharges']

for col in encode_cols:
    if col in ['tenure', 'MonthlyCharges']:
        # 수치형 변수: orig 기준으로 10개 구간 분할 및 경계값(bins_edge) 추출
        orig_bins, bins_edge = pd.qcut(orig[col], 10, duplicates='drop', retbins=True)

        # 구간별 평균 이탈률 계산
        orig_val = orig.groupby(orig_bins, observed=False)['Churn'].mean()
        orig_val.index = orig_val.index.astype(str)

        # 데이터가 범위를 벗어나 NaN이 되지 않도록 양끝 경계를 무한대로 확장
        bins_edge[0] = -np.inf
        bins_edge[-1] = np.inf

        # 추출한 절대 경계값을 기준으로 df/test 데이터를 자르고 확률 매핑
        df_bins = pd.cut(df[col], bins=bins_edge).astype(str)
        df[f'ORIG_{col}'] = df_bins.map(orig_val).astype(float)

        test_bins = pd.cut(test[col], bins=bins_edge).astype(str)
        test[f'ORIG_{col}'] = test_bins.map(orig_val).astype(float)

    else:
        # 범주형 변수: 카테고리별 이탈률 딕셔너리 생성 후 매핑
        tmp = orig.groupby(col)['Churn'].mean().to_dict()
        df[f'ORIG_{col}'] = df[col].map(tmp).astype(float)
        test[f'ORIG_{col}'] = test[col].map(tmp).astype(float)

In [25]:
# 최종 학습/예측 데이터 세팅
X_train = df.drop(columns=['id', 'Churn'])
y_train = df['Churn']
X_test = test.drop(columns=['id'])

# Tree 모델 학습을 위해 범주형 변수를 category 타입으로 변환
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()
for col in cat_features:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print("데이터 전처리 및 Target Encoding 완료.")
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

데이터 전처리 및 Target Encoding 완료.
Train shape: (594194, 26), Test shape: (254655, 26)


In [26]:
# X_test의 범주형 변수를 X_train의 카테고리 기준으로 강제 동기화
for col in cat_features:
    X_test[col] = pd.Categorical(X_test[col], categories=X_train[col].cat.categories)

In [27]:
# --- 1단계 모델용 변수 분류 (Logistic Regression 전처리용) ---
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['category', 'object']).columns.tolist()

# 수치형 변수 파이프라인: 결측치를 평균(mean)으로 채우고 스케일링
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# 범주형 변수 파이프라인: 결측치를 최빈값(most_frequent)으로 채우고 원핫인코딩
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 전처리 파이프라인 결합
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numeric_features),
        ('cat', cat_transformer, categorical_features)
    ])

# 3. GPU Ensemble (Fast & Deep 모델) - CatBoost 제외
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 이번엔 모델이 2개뿐입니다!
oof_preds = np.zeros((len(X_train), 2))
test_preds = np.zeros((len(X_test), 2))

print("🚀 최고 점수(Depth 7) 탈환을 위한 Fast Training 시작...\n")

for i, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"--- Processing Fold {i+1} ---")
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    # M1: XGBoost (깊이는 7로 복구, 학습률은 0.03으로 올려서 고속화)
    xgb_model = xgb.XGBClassifier(
        n_estimators=2000,
        learning_rate=0.03,  # 기존 0.005 -> 0.03으로 올려서 6배 빠르게 수렴
        max_depth=7,
        device='cuda',
        tree_method='hist',
        enable_categorical=True,
        eval_metric='auc',
        early_stopping_rounds=100,
        random_state=42
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)

    # M2: LightGBM (깊이 7 복구, 학습률 상향)
    lgb_model = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=7,
        num_leaves=63, # 2^6 - 1 정도로 설정
        device_type='gpu',
        random_state=42
    )
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(100, verbose=False)])

    # OOF 예측값 저장
    oof_preds[val_idx, 0] = xgb_model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx, 1] = lgb_model.predict_proba(X_val)[:, 1]

    # Test 데이터 예측 누적
    test_preds[:, 0] += xgb_model.predict_proba(X_test)[:, 1] / skf.n_splits
    test_preds[:, 1] += lgb_model.predict_proba(X_test)[:, 1] / skf.n_splits

print("\n--- OOF AUC Scores ---")
print(f"XGBoost (Fast) OOF AUC: {roc_auc_score(y_train, oof_preds[:, 0]):.5f}")
print(f"LightGBM (Fast) OOF AUC: {roc_auc_score(y_train, oof_preds[:, 1]):.5f}")

# 4. Final Stacking Meta-Learner (Ridge)
meta_model = Ridge(alpha=1.0, random_state=42)
meta_model.fit(oof_preds, y_train)

print("\n--- Meta-Learner Weights ---")
print(f"XGBoost: {meta_model.coef_[0]:.4f}")
print(f"LightGBM: {meta_model.coef_[1]:.4f}")

final_probs = np.clip(meta_model.predict(test_preds), 0, 1)
meta_oof_preds = np.clip(meta_model.predict(oof_preds), 0, 1)

print(f"\n🏆 Final Stacking OOF AUC: {roc_auc_score(y_train, meta_oof_preds):.5f}")

submission = pd.DataFrame({'id': test_raw['id'], 'Churn': final_probs})
submission.to_csv('submission_4.csv', index=False)
print("\n'submission_4.csv' 완료!")

🚀 최고 점수(Depth 7) 탈환을 위한 Fast Training 시작...

--- Processing Fold 1 ---
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 936
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 26
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 23 dense feature groups (10.88 MB) transferred to GPU in 0.023659 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 